In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as plt

In [3]:

import glob

# Get all CSV files from data folder
files = glob.glob('../data/*.csv')

# Load and combine
df = pd.concat([pd.read_csv(file) for file in files], ignore_index=True)

df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-12-01 00:00:00 UTC,remove_from_cart,5712790,1487580005268456287,NaN,f.o.x,6.27,576802932,51d85cb0-897f-48d2-918b-ad63965c12dc
1,2019-12-01 00:00:00 UTC,view,5764655,1487580005411062629,NaN,cnd,29.05,412120092,8adff31e-2051-4894-9758-224bfa8aec18
2,2019-12-01 00:00:02 UTC,cart,4958,1487580009471148064,NaN,runail,1.19,494077766,c99a50e8-2fac-4c4d-89ec-41c05f114554
3,2019-12-01 00:00:05 UTC,view,5848413,1487580007675986893,NaN,freedecor,0.79,348405118,722ffea5-73c0-4924-8e8f-371ff8031af4
4,2019-12-01 00:00:07 UTC,view,5824148,1487580005511725929,NaN,NaN,5.56,576005683,28172809-7e4a-45ce-bab0-5efa90117cd5


In [4]:
df = pd.concat([
    pd.read_csv(file, usecols=[
        'event_time','event_type','product_id',
        'category_code','brand','price','user_id'
    ])
    for file in files
], ignore_index=True)

In [5]:
df['event_time'] = pd.to_datetime(df['event_time'])

df['category_code'] = df['category_code'].fillna('unknown')
df['brand'] = df['brand'].fillna('unknown')

df = df.drop_duplicates()

In [6]:
df.shape

(11624737, 7)

In [7]:
df['event_type'].value_counts()

event_type
view                5663305
cart                3405489
remove_from_cart    1775327
purchase             780616
Name: count, dtype: int64

In [8]:
#Funnel analysis
funnel = df['event_type'].value_counts()

views = funnel.get('view', 0)
cart = funnel.get('cart', 0)
purchase = funnel.get('purchase', 0)
remove = funnel.get('remove_from_cart', 0)

print("View → Cart:", (cart/views)*100)
print("Cart → Purchase:", (purchase/cart)*100)
print("Cart Removal Rate:", (remove/cart)*100)

View → Cart: 60.132537449422195
Cart → Purchase: 22.922288106054665
Cart Removal Rate: 52.13133855372899


In [9]:
#Revenue
revenue = df[df['event_type']=='purchase']['price'].sum()
print("Total Revenue:", revenue)

Total Revenue: 3817894.210000001


In [10]:
#TOP PRODUCTS & BRANDS
df[df['event_type']=='purchase']['product_id'].value_counts().head(10)

df[df['event_type']=='purchase']['brand'].value_counts().head(10)

brand
unknown      330155
runail        66115
irisk         42566
masura        29704
bpw.style     29505
grattol       28222
ingarden      17128
freedecor     11119
estel         10809
uno           10715
Name: count, dtype: int64

In [11]:
#TIME ANALYSIS
df['hour'] = df['event_time'].dt.hour

df['hour'].value_counts().sort_index()

hour
0     113755
1     104447
2     115369
3     156851
4     226347
5     347215
6     468420
7     552573
8     612329
9     640151
10    682166
11    699567
12    700354
13    659349
14    616224
15    587292
16    611563
17    656677
18    723029
19    759204
20    682860
21    471124
22    278189
23    159682
Name: count, dtype: int64

In [12]:
#CART ABANDONMENT
cart_users = set(df[df['event_type']=='cart']['user_id'])
purchase_users = set(df[df['event_type']=='purchase']['user_id'])

abandon_rate = ((len(cart_users - purchase_users)) / len(cart_users)) * 100

print("Cart Abandonment Rate:", round(abandon_rate,2), "%")

Cart Abandonment Rate: 73.62 %


In [13]:
#cohort analysis
df['month'] = df['event_time'].dt.to_period('M')

df['cohort'] = df.groupby('user_id')['event_time'] \
                 .transform('min') \
                 .dt.to_period('M')

C:\Users\hp\AppData\Local\Temp\ipykernel_17008\2117414451.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['month'] = df['event_time'].dt.to_period('M')
C:\Users\hp\AppData\Local\Temp\ipykernel_17008\2117414451.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['cohort'] = df.groupby('user_id')['event_time'] \


In [14]:
df['event_time'] = df['event_time'].dt.tz_localize(None)

df['month'] = df['event_time'].dt.to_period('M')

In [15]:
cohort_data = df.groupby(['cohort','month'])['user_id'] \
                .nunique() \
                .reset_index()

In [16]:
cohort_pivot = cohort_data.pivot(
    index='cohort',
    columns='month',
    values='user_id'
)

cohort_pivot

month,2019-10,2019-11,2019-12
cohort,,,
2019-10,399664.0,54796.0,36346.0
2019-11,NaN,313436.0,34347.0
2019-12,NaN,NaN,299461.0


In [19]:
retention = cohort_pivot.copy()

for i in range(len(retention)):
    retention.iloc[i] = retention.iloc[i] / retention.iloc[i].dropna().iloc[0]

retention.round(3)

month,2019-10,2019-11,2019-12
cohort,,,
2019-10,1.0,0.137,0.091
2019-11,NaN,1.000,0.110
2019-12,NaN,NaN,1.000


- High drop-off from cart to purchase indicates checkout friction.
- Cart removal events suggest pricing or trust issues.
- Peak activity during evening hours indicates best time for campaigns.
- Few products and brands contribute majority of revenue.

In [20]:
df['event_type'].value_counts()

event_type
view                5663305
cart                3405489
remove_from_cart    1775327
purchase             780616
Name: count, dtype: int64

In [21]:
df[df['event_type']=='purchase']['price'].sum()

3817894.210000001

In [22]:
df[df['event_type']=='purchase']['product_id'] \
    .value_counts().head(10)

product_id
5809910    4108
5854897    2651
5700037    2189
5802432    2159
5751422    2126
5304       1938
5815662    1885
5809912    1810
5751383    1803
5833330    1766
Name: count, dtype: int64